# 03 — Visualization: Visual Evidence for Agenda Distortion

Produces the visual evidence stack for H1. All plots use the same UMAP projection of the combined corpus to ensure comparability.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "agenda_distortion" / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

from bertopic import BERTopic
from umap import UMAP

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

In [ ]:
# Load document-topic assignments
alt_doc_topics = pd.read_csv(OUTPUT_DIR / "alt_media_doc_topics.csv")
ms_doc_topics = pd.read_csv(OUTPUT_DIR / "mainstream_doc_topics.csv")

# Combined frame for UMAP
alt_doc_topics["media_type"] = "Alt Media"
ms_doc_topics["media_type"] = "Mainstream"
combined = pd.concat([alt_doc_topics, ms_doc_topics], ignore_index=True)

print(f"Combined: {len(combined):,} documents")
print(f"  Alt media:  {len(alt_doc_topics):,}")
print(f"  Mainstream: {len(ms_doc_topics):,}")

# Load models for embedding extraction
alt_model = BERTopic.load(OUTPUT_DIR / "alt_media_model", embedding_model=EMBEDDING_MODEL)
ms_model = BERTopic.load(OUTPUT_DIR / "mainstream_model", embedding_model=EMBEDDING_MODEL)

## UMAP of All Document Embeddings — Colored by Media Type

Fit UMAP on the combined corpus. Color by media type (alt vs mainstream). A clean separation in UMAP space is visual evidence of divergent topic worlds.

**If no separation is visible**: the embedding model may be too generic — consider domain-fine-tuned sentence transformers.

In [ ]:
# Extract embeddings for all documents
all_docs = combined["document"].tolist()
# Use the alt model's embedding extractor (same underlying model)
all_embeddings = alt_model._extract_embeddings(all_docs, method="document")

# Fit 2D UMAP
reducer_2d = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
coords_2d = reducer_2d.fit_transform(all_embeddings)

combined["umap_x"] = coords_2d[:, 0]
combined["umap_y"] = coords_2d[:, 1]
print(f"UMAP fitted: {coords_2d.shape[0]} documents → 2D")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10), dpi=150)
fig.patch.set_facecolor("white")
ax.set_facecolor("#F7F7F7")

colors = {"Mainstream": "#4878CF", "Alt Media": "#B22222"}

for media_type in ["Mainstream", "Alt Media"]:
    subset = combined.loc[combined["media_type"] == media_type]
    ax.scatter(
        subset["umap_x"],
        subset["umap_y"],
        c=colors[media_type],
        s=3,
        alpha=0.3,
        label=media_type,
        linewidths=0,
        rasterized=True,
    )

ax.legend(fontsize=11, markerscale=5)
ax.set_title("UMAP Projection — Alt Media vs Mainstream", fontsize=14, pad=14)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "umap_media_type.pdf", bbox_inches="tight")
plt.show()

## UMAP Colored by Topic

Same projection, now colored by assigned topic. Look for topic clusters that are exclusively or predominantly one media type.

In [ ]:
# Use the per-corpus topic assignments
# Since topics are from different models, we label them as "Alt-{id}" or "MS-{id}"
combined["topic_label"] = combined.apply(
    lambda row: f"{row['media_type'][:3]}-{row['topic']}" if row["topic"] != -1 else "Outlier",
    axis=1,
)

fig, ax = plt.subplots(figsize=(14, 10), dpi=150)
fig.patch.set_facecolor("white")
ax.set_facecolor("#F7F7F7")

# Outliers in grey
outliers = combined.loc[combined["topic"] == -1]
ax.scatter(
    outliers["umap_x"], outliers["umap_y"],
    c="#d9d9d9", s=2, alpha=0.15, linewidths=0, rasterized=True, label="Outlier",
)

# Non-outliers colored by topic
non_outlier = combined.loc[combined["topic"] != -1]
scatter = ax.scatter(
    non_outlier["umap_x"], non_outlier["umap_y"],
    c=non_outlier["topic"], cmap="tab20", s=3, alpha=0.35, linewidths=0, rasterized=True,
)

ax.set_title("UMAP Projection — Colored by Topic Assignment", fontsize=14, pad=14)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "umap_topics.pdf", bbox_inches="tight")
plt.show()

## Prevalence Heatmap

Topics (rows) x media type (columns), cell = normalized prevalence. Identifies which topics are agenda priorities for alt vs mainstream.

In [ ]:
with open(OUTPUT_DIR / "alt_media_topic_words.json") as f:
    alt_word_lists = json.load(f)
with open(OUTPUT_DIR / "mainstream_topic_words.json") as f:
    ms_word_lists = json.load(f)

# Top 15 topics by prevalence in each corpus
alt_non_outlier = alt_doc_topics.loc[alt_doc_topics["topic"] != -1]
ms_non_outlier = ms_doc_topics.loc[ms_doc_topics["topic"] != -1]

alt_prev = alt_non_outlier["topic"].value_counts(normalize=True).head(15)
ms_prev = ms_non_outlier["topic"].value_counts(normalize=True).head(15)

# Build heatmap data
all_top_topics = sorted(set(alt_prev.index.tolist() + ms_prev.index.tolist()))

heatmap_data = pd.DataFrame({
    "Alt Media": [alt_prev.get(t, 0.0) for t in all_top_topics],
    "Mainstream": [ms_prev.get(t, 0.0) for t in all_top_topics],
}, index=[
    ", ".join(alt_word_lists.get(str(t), ms_word_lists.get(str(t), [str(t)]))[:4])
    for t in all_top_topics
])

fig, ax = plt.subplots(figsize=(8, max(6, len(all_top_topics) * 0.4)), dpi=150)
fig.patch.set_facecolor("white")
im = ax.imshow(heatmap_data.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(2))
ax.set_xticklabels(["Alt Media", "Mainstream"])
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index, fontsize=8)
ax.set_title("Topic Prevalence: Alt Media vs Mainstream", fontsize=13, pad=12)
fig.colorbar(im, ax=ax, label="Normalized prevalence", fraction=0.05)

# Annotate cells
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        val = heatmap_data.iat[i, j]
        if val > 0.005:
            ax.text(j, i, f"{val:.2%}", ha="center", va="center", fontsize=7,
                    color="white" if val > 0.05 else "black")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "prevalence_heatmap.pdf", bbox_inches="tight")
plt.show()

## Top-K Topic Distribution Overlap

Side-by-side bar chart of top-15 topics by prevalence in each corpus. Visual complement to the JSD score.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=150, sharey=False)
fig.patch.set_facecolor("white")

for ax, (prev, word_lists, title, color) in zip(axes, [
    (alt_prev, alt_word_lists, "Alt Media — Top 15 Topics", "#B22222"),
    (ms_prev, ms_word_lists, "Mainstream — Top 15 Topics", "#4878CF"),
]):
    labels = [", ".join(word_lists.get(str(t), [str(t)])[:3]) for t in prev.index]
    y_pos = range(len(prev))
    ax.barh(y_pos, prev.values, color=color, alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Normalized prevalence")
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_facecolor("#FAFAFA")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(axis="x", color="#E0E0E0", linewidth=0.7)

fig.suptitle("Topic Distribution Comparison", fontsize=14, weight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "top_k_comparison.pdf", bbox_inches="tight")
plt.show()

## Visual Sanity Check

Answer these questions after running all cells:

1. **Does UMAP show meaningful separation between alt and mainstream?** (Y/N, describe)
2. **Are there topic clusters exclusive to alt media?** Which topics?
3. **Does the prevalence heatmap show systematic over-representation of certain themes in alt media?**
4. **If visualizations are noisy/uninterpretable**: increase UMAP `n_neighbors`, or go back to notebook 01 and increase `min_topic_size` for cleaner topics.